# Overall Experiment

This notebook is the **analysis / plotting layer** on top of the benchmark CSVs produced by the `hedge-automata` experiment notebook. It does not run any automata itself — it loads the `time_ms` results already exported to CSV (e.g. `hedge_dfa_acceptor_benchmark.csv`, `hedge_acceptor_benchmark.csv`), merges the DFA-based and regex-based acceptor results, and produces publication-style comparison plots (runtime vs. file size, throughput vs. file size, etc.).

This section compares the **DFA-based streaming acceptor** against the **regex-based streaming acceptor** on the *accepting* dataset (files that pass validation), across the full range of generated file sizes.

**What this cell does:**
1. Loads two benchmark CSVs — one produced by the DFA-based acceptor (`hedge_dfa_acceptor_benchmark.csv`) and one produced by the regex-based acceptor (`hedge_acceptor_benchmark.csv`).
2. Renames each CSV's `time_ms` column so they don't collide (`time_ms_dfa` / `time_ms_acceptor`), then inner-joins the two on `file`, `size_bytes`, `accepted` so each row holds both acceptors' timings for the same file.
3. Adds a `size_mb` column and saves the combined table to `merged_benchmark.csv`.
4. Reshapes the merged table into **long format** (`melt`) so seaborn can plot one line per method.
5. Defines `make_plot(...)`, a helper that draws a log-log line plot of time vs. file size for a given subset of the data, with consistent "publication" styling (serif font, custom color palette, saved as both PNG and PDF).
6. Calls `make_plot` three times to produce: a plot over **all files**, a plot restricted to **small files (<100 MB)**, and a plot restricted to **large files (≥100 MB)** — letting you inspect scaling behavior at both ends of the size range without one extreme squashing the other in a single log-log chart.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ---------------------------------------------------------------
# Load & merge
# ---------------------------------------------------------------
dfa = pd.read_csv('/content/hedge_dfa_acceptor_benchmark.csv')
acc = pd.read_csv('/content/hedge_acceptor_benchmark.csv')

key_cols = ['file', 'size_bytes', 'accepted']
dfa = dfa.rename(columns={'time_ms': 'time_ms_dfa'})
acc = acc.rename(columns={'time_ms': 'time_ms_acceptor'})

merged = pd.merge(dfa, acc, on=key_cols, how='inner', validate='one_to_one')
merged['size_mb'] = merged['size_bytes'] / (1024 ** 2)
merged = merged.sort_values('size_mb').reset_index(drop=True)

merged.to_csv('merged_benchmark.csv', index=False)
print(merged.shape)
print(merged.head())

# Long format for seaborn (one row per method)
long_df = merged.melt(
    id_vars=['file', 'size_bytes', 'accepted', 'size_mb'],
    value_vars=['time_ms_dfa', 'time_ms_acceptor'],
    var_name='method',
    value_name='time_ms'
)
label_map = {'time_ms_dfa': 'Dfa-based', 'time_ms_acceptor': 'Regex-based'}
long_df['method'] = long_df['method'].map(label_map)

# ---------------------------------------------------------------
# Publication style
# ---------------------------------------------------------------
sns.set_theme(context='paper', style='whitegrid', font_scale=1.3)
plt.rcParams.update({
    'font.family': 'serif',
    'axes.edgecolor': '0.3',
    'axes.linewidth': 1.0,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.4,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'savefig.dpi': 300,
    'figure.dpi': 150,
})
palette = {'Dfa-based': '#1f77b4', 'Regex-based': '#d62728'}


def make_plot(df_subset, title, fname, size_threshold_label):
    fig, ax = plt.subplots(figsize=(7, 5))
    df_sorted = df_subset.sort_values('size_mb')
    sns.lineplot(
        data=df_sorted, x='size_mb', y='time_ms', hue='method',
        style='method', palette=palette, markers=True, dashes=False,
        markersize=5, linewidth=1.3, alpha=0.9, ax=ax,
        markeredgecolor='white', markeredgewidth=0.4,
    )
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('File size (MB, log scale)')
    ax.set_ylabel('Time (ms, log scale)')
    ax.set_title(title, fontsize=13, weight='bold')
    ax.legend(title='Acceptor', loc='upper left')

    def _fmt(x, pos):
        if x >= 1:
            return f'{x:g}'
        return f'{x:g}'
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt))
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())
    fig.tight_layout()
    fig.savefig(f'{fname}.png', bbox_inches='tight')
    fig.savefig(f'{fname}.pdf', bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {fname} ({size_threshold_label}): n={len(df_subset)//2} points/method')


# 1. All data
make_plot(long_df, 'Benchmark Time vs. File Size (All Files)', 'plot_all', 'all')

# 2. Small files (<100MB)
small = long_df[long_df['size_mb'] < 100]
make_plot(small, 'Benchmark Time vs. File Size (Small Files, <100 MB)', 'plot_small', '<100MB')

# 3. Big files (>=100MB)
big = long_df[long_df['size_mb'] >= 100]
make_plot(big, 'Benchmark Time vs. File Size (Large Files, ≥100 MB)', 'plot_large', '>=100MB')

print('Done')

(56, 6)
            file  size_bytes  accepted  time_ms_dfa  time_ms_acceptor  \
0    vsmall.json          90      True     0.067470          0.061380   
1     small.json         191      True     0.069240          0.070340   
2    bsmall.json      104340      True     3.057803          3.087693   
3  large_11.json      285085      True     7.757878          7.840308   
4  large_21.json      318942      True     8.930379          8.752879   

    size_mb  
0  0.000086  
1  0.000182  
2  0.099506  
3  0.271878  
4  0.304167  
Saved plot_all (all): n=56 points/method
Saved plot_small (<100MB): n=28 points/method
Saved plot_large (>=100MB): n=28 points/method
Done


# Rejection Experiment

This section repeats the same DFA-vs-regex comparison, but on **rejecting** inputs — malformed/invalid JSON files that the hedge automaton should refuse. This matters because the DFA-based acceptor is designed to fail fast (`HedgeRejected`) as soon as it can prove a file is invalid, so rejection performance can look very different from acceptance performance, especially on large files where an early rejection avoids reading the rest of the file.

**What this cell does:** Same merge-and-plot pattern as the Overall Experiment cell, but reading from `experiments/rejection/dfa_reject.csv` and `experiments/rejection/regex_reject.csv`. It merges the two acceptors' timings on rejected files, saves `merged_reject.csv`, and produces a single (linear-scale, not log-log) time-vs-size plot saved as `plot_reject.png` / `.pdf`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------
# Load & merge
# ---------------------------------------------------------------
dfa = pd.read_csv('/content/experiments/rejection/dfa_reject.csv')
rgx = pd.read_csv('/content/experiments/rejection/regex_reject.csv')

key_cols = ['file', 'size_bytes', 'accepted']
dfa = dfa.rename(columns={'time_ms': 'time_ms_dfa'})
rgx = rgx.rename(columns={'time_ms': 'time_ms_regex'})

merged = pd.merge(dfa, rgx, on=key_cols, how='inner', validate='one_to_one')
merged['size_mb'] = merged['size_bytes'] / (1024 ** 2)
merged = merged.sort_values('size_mb').reset_index(drop=True)
merged.to_csv('/content/experiments/rejection/merged_reject.csv', index=False)
print(merged)

long_df = merged.melt(
    id_vars=['file', 'size_bytes', 'accepted', 'size_mb'],
    value_vars=['time_ms_dfa', 'time_ms_regex'],
    var_name='method', value_name='time_ms'
)
label_map = {'time_ms_dfa': 'DFA-based', 'time_ms_regex': 'Regex-based'}
long_df['method'] = long_df['method'].map(label_map)
long_df = long_df.sort_values('size_mb')

# ---------------------------------------------------------------
# Publication style (matching earlier plots)
# ---------------------------------------------------------------
sns.set_theme(context='paper', style='whitegrid', font_scale=1.3)
plt.rcParams.update({
    'font.family': 'serif',
    'axes.edgecolor': '0.3',
    'axes.linewidth': 1.0,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.4,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'savefig.dpi': 300,
    'figure.dpi': 150,
})
palette = {'DFA-based': '#1f77b4', 'Regex-based': '#d62728'}

fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(
    data=long_df, x='size_mb', y='time_ms', hue='method',
    style='method', palette=palette, markers=True, dashes=False,
    markersize=7, linewidth=1.5, alpha=0.9, ax=ax,
    markeredgecolor='white', markeredgewidth=0.6,
)
ax.set_xlabel('File size (MB)')
ax.set_ylabel('Time (ms)')
ax.set_title('Rejection Time vs. File Size', fontsize=13, weight='bold')
ax.legend(title='Method', loc='upper left')
fig.tight_layout()
fig.savefig('/content/experiments/rejection/plot_reject.png', bbox_inches='tight')
fig.savefig('/content/experiments/rejection/plot_reject.pdf', bbox_inches='tight')
plt.close(fig)
print('Saved plot_reject')

                                       file  size_bytes  accepted  \
0   employees_5-level_100-MB_formatted.json   107979900     False   
1  employees_10-level_200-MB_formatted.json   215303381     False   
2   employees_5-level_300-MB_formatted.json   323935493     False   
3  employees_10-level_500-MB_formatted.json   538246875     False   
4    employees_10-level_1-GB_formatted.json  1102324379     False   

    time_ms_dfa  time_ms_regex      size_mb  
0   1245.346520    1319.468560   102.977657  
1   2291.681461    2424.933335   205.329305  
2   3794.723817    3927.939381   308.928960  
3   5646.933024    6045.901683   513.312221  
4  11406.649660   11595.454122  1051.258449  
Saved plot_reject


# By-Depth Experiment

Here file size is no longer the only variable of interest — this section looks at how acceptor runtime scales as the **nesting depth** of the generated JSON is increased (holding `members_per_object` and `array_size` fixed), using the `depth`-sweep dataset produced by `generate_experiments(mode="depth", ...)` in the generation notebook. Deeper trees stress the automaton's recursion / stack depth rather than its raw throughput.

**What this cell does:** Loads and merges the DFA and regex acceptor timings from `experiments/by_depth/dfa_by_depth.csv` and `experiments/by_depth/regex_by_depth.csv`, saves the merged table, and plots time vs. file size for this depth-varying dataset (`plot_reject.png/.pdf` under `by_depth/`). Note the file size still varies here as a side effect of depth changing, so the x-axis is still size in MB, but the underlying independent variable being swept is depth.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------
# Load & merge
# ---------------------------------------------------------------
dfa = pd.read_csv('/content/experiments/by_depth/dfa_by_depth.csv')
rgx = pd.read_csv('/content/experiments/by_depth/regex_by_depth.csv')

key_cols = ['file', 'size_bytes', 'accepted']
dfa = dfa.rename(columns={'time_ms': 'time_ms_dfa'})
rgx = rgx.rename(columns={'time_ms': 'time_ms_regex'})

merged = pd.merge(dfa, rgx, on=key_cols, how='inner', validate='one_to_one')
merged['size_mb'] = merged['size_bytes'] / (1024 ** 2)
merged = merged.sort_values('size_mb').reset_index(drop=True)
merged.to_csv('/content/experiments/by_depth/merged_reject.csv', index=False)
print(merged)

long_df = merged.melt(
    id_vars=['file', 'size_bytes', 'accepted', 'size_mb'],
    value_vars=['time_ms_dfa', 'time_ms_regex'],
    var_name='method', value_name='time_ms'
)
label_map = {'time_ms_dfa': 'DFA-based', 'time_ms_regex': 'Regex-based'}
long_df['method'] = long_df['method'].map(label_map)
long_df = long_df.sort_values('size_mb')

# ---------------------------------------------------------------
# Publication style (matching earlier plots)
# ---------------------------------------------------------------
sns.set_theme(context='paper', style='whitegrid', font_scale=1.3)
plt.rcParams.update({
    'font.family': 'serif',
    'axes.edgecolor': '0.3',
    'axes.linewidth': 1.0,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.4,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'savefig.dpi': 300,
    'figure.dpi': 150,
})
palette = {'DFA-based': '#1f77b4', 'Regex-based': '#d62728'}

fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(
    data=long_df, x='size_mb', y='time_ms', hue='method',
    style='method', palette=palette, markers=True, dashes=False,
    markersize=7, linewidth=1.5, alpha=0.9, ax=ax,
    markeredgecolor='white', markeredgewidth=0.6,
)
ax.set_xlabel('File size (MB)')
ax.set_ylabel('Time (ms)')
ax.set_title('By-depth Time vs. File Size', fontsize=13, weight='bold')
ax.legend(title='Method', loc='upper left')
fig.tight_layout()
fig.savefig('/content/experiments/by_depth/plot_reject.png', bbox_inches='tight')
fig.savefig('/content/experiments/by_depth/plot_reject.pdf', bbox_inches='tight')
plt.close(fig)
print('Saved plot_reject')

                  file  size_bytes  accepted    time_ms_dfa  time_ms_regex  \
0   depth_5_m6_a6.json      104137      True       3.186923       2.989933   
1   depth_6_m6_a6.json      703948      True      21.343870      20.610581   
2   depth_7_m6_a6.json     4398782      True     127.637008     121.589442   
3   depth_8_m6_a6.json    18530893      True     531.862058     502.580659   
4   depth_9_m6_a6.json   104982810      True    2903.860712    2780.654816   
5  depth_10_m6_a6.json   681692483      True   17995.926278   17130.929845   
6  depth_11_m6_a6.json  4649395752      True  121497.611487  115203.899947   

       size_mb  
0     0.099313  
1     0.671337  
2     4.195005  
3    17.672437  
4   100.119410  
5   650.112613  
6  4434.009315  
Saved plot_reject


# By-Members Experiment

Same idea, but sweeping `members_per_object` (the branching factor of JSON *objects*) while holding depth and array size fixed. This stresses the horizontal-language matching for the `object` transition rule, since a wider object means a longer sequence of child states to match against `(qV|qO|qA)*`.

**What this cell does:** Loads and merges `experiments/by_members/dfa_by_members.csv` and `regex_by_members.csv`, saves the merged table, and plots time vs. file size for the members-varying dataset (`plot_reject.png/.pdf` under `by_members/`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------
# Load & merge
# ---------------------------------------------------------------
dfa = pd.read_csv('/content/experiments/by_members/dfa_by_members.csv')
rgx = pd.read_csv('/content/experiments/by_members/regex_by_members.csv')

key_cols = ['file', 'size_bytes', 'accepted']
dfa = dfa.rename(columns={'time_ms': 'time_ms_dfa'})
rgx = rgx.rename(columns={'time_ms': 'time_ms_regex'})

merged = pd.merge(dfa, rgx, on=key_cols, how='inner', validate='one_to_one')
merged['size_mb'] = merged['size_bytes'] / (1024 ** 2)
merged = merged.sort_values('size_mb').reset_index(drop=True)
merged.to_csv('/content/experiments/by_members/merged_reject.csv', index=False)
print(merged)

long_df = merged.melt(
    id_vars=['file', 'size_bytes', 'accepted', 'size_mb'],
    value_vars=['time_ms_dfa', 'time_ms_regex'],
    var_name='method', value_name='time_ms'
)
label_map = {'time_ms_dfa': 'DFA-based', 'time_ms_regex': 'Regex-based'}
long_df['method'] = long_df['method'].map(label_map)
long_df = long_df.sort_values('size_mb')

# ---------------------------------------------------------------
# Publication style (matching earlier plots)
# ---------------------------------------------------------------
sns.set_theme(context='paper', style='whitegrid', font_scale=1.3)
plt.rcParams.update({
    'font.family': 'serif',
    'axes.edgecolor': '0.3',
    'axes.linewidth': 1.0,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.4,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'savefig.dpi': 300,
    'figure.dpi': 150,
})
palette = {'DFA-based': '#1f77b4', 'Regex-based': '#d62728'}

fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(
    data=long_df, x='size_mb', y='time_ms', hue='method',
    style='method', palette=palette, markers=True, dashes=False,
    markersize=7, linewidth=1.5, alpha=0.9, ax=ax,
    markeredgecolor='white', markeredgewidth=0.6,
)
ax.set_xlabel('File size (MB)')
ax.set_ylabel('Time (ms)')
ax.set_title('By_members Time vs. File Size', fontsize=13, weight='bold')
ax.legend(title='Method', loc='upper left')
fig.tight_layout()
fig.savefig('/content/experiments/by_members/plot_reject.png', bbox_inches='tight')
fig.savefig('/content/experiments/by_members/plot_reject.pdf', bbox_inches='tight')
plt.close(fig)
print('Saved plot_reject')

                                file  size_bytes  accepted   time_ms_dfa  \
0    depth_7_members_5_arrays_6.json     1726626      True     51.495868   
1    depth_7_members_6_arrays_6.json     4068852      True    111.606321   
2    depth_7_members_7_arrays_6.json     9086473      True    238.703430   
3    depth_7_members_8_arrays_6.json    16493414      True    426.913977   
4    depth_7_members_9_arrays_6.json    28505913      True    726.108457   
5   depth_7_members_10_arrays_6.json    47229510      True   1101.221828   
6   depth_7_members_11_arrays_6.json    80779870      True   1811.954611   
7   depth_7_members_12_arrays_6.json   127134991      True   2798.745203   
8   depth_7_members_13_arrays_6.json   220725636      True   5059.693716   
9   depth_7_members_14_arrays_6.json   254211999      True   5797.169703   
10  depth_7_members_15_arrays_6.json   410655519      True   9244.946881   
11  depth_7_members_16_arrays_6.json   707011775      True  15911.715291   
12  depth_7_

# By-Arrays Experiment

Same idea again, this time sweeping `array_size` (the branching factor of JSON *arrays*), which stresses the `array` transition rule's horizontal language `(qO|qV)*`.

**What this cell does:** Loads and merges `experiments/by_arrays/dfa_by_array.csv` and `regex_by_array.csv`, saves the merged table, and plots time vs. file size for the arrays-varying dataset (`plot_reject.png/.pdf` under `by_arrays/`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------
# Load & merge
# ---------------------------------------------------------------
dfa = pd.read_csv('/content/experiments/by_arrays/dfa_by_array.csv')
rgx = pd.read_csv('/content/experiments/by_arrays/regex_by_array.csv')

key_cols = ['file', 'size_bytes', 'accepted']
dfa = dfa.rename(columns={'time_ms': 'time_ms_dfa'})
rgx = rgx.rename(columns={'time_ms': 'time_ms_regex'})

merged = pd.merge(dfa, rgx, on=key_cols, how='inner', validate='one_to_one')
merged['size_mb'] = merged['size_bytes'] / (1024 ** 2)
merged = merged.sort_values('size_mb').reset_index(drop=True)
merged.to_csv('/content/experiments/by_arrays/merged_reject.csv', index=False)
print(merged)

long_df = merged.melt(
    id_vars=['file', 'size_bytes', 'accepted', 'size_mb'],
    value_vars=['time_ms_dfa', 'time_ms_regex'],
    var_name='method', value_name='time_ms'
)
label_map = {'time_ms_dfa': 'DFA-based', 'time_ms_regex': 'Regex-based'}
long_df['method'] = long_df['method'].map(label_map)
long_df = long_df.sort_values('size_mb')

# ---------------------------------------------------------------
# Publication style (matching earlier plots)
# ---------------------------------------------------------------
sns.set_theme(context='paper', style='whitegrid', font_scale=1.3)
plt.rcParams.update({
    'font.family': 'serif',
    'axes.edgecolor': '0.3',
    'axes.linewidth': 1.0,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.4,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'savefig.dpi': 300,
    'figure.dpi': 150,
})
palette = {'DFA-based': '#1f77b4', 'Regex-based': '#d62728'}

fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(
    data=long_df, x='size_mb', y='time_ms', hue='method',
    style='method', palette=palette, markers=True, dashes=False,
    markersize=7, linewidth=1.5, alpha=0.9, ax=ax,
    markeredgecolor='white', markeredgewidth=0.6,
)
ax.set_xlabel('File size (MB)')
ax.set_ylabel('Time (ms)')
ax.set_title('By-arrays Time vs. File Size', fontsize=13, weight='bold')
ax.legend(title='Method', loc='upper left')
fig.tight_layout()
fig.savefig('/content/experiments/by_arrays/plot_reject.png', bbox_inches='tight')
fig.savefig('/content/experiments/by_arrays/plot_reject.pdf', bbox_inches='tight')
plt.close(fig)
print('Saved plot_reject')

                                file  size_bytes  accepted   time_ms_dfa  \
0    depth_7_members_6_arrays_5.json     2758937      True     79.758617   
1    depth_7_members_6_arrays_6.json     3409340      True     97.670191   
2    depth_7_members_6_arrays_7.json     5754494      True    164.585299   
3    depth_7_members_6_arrays_8.json     7610342      True    214.962200   
4    depth_7_members_6_arrays_9.json     8800394      True    249.013231   
5   depth_7_members_6_arrays_10.json    11105727      True    323.037360   
6   depth_7_members_6_arrays_11.json    11322962      True    293.307481   
7   depth_7_members_6_arrays_12.json    20279927      True    528.767170   
8   depth_7_members_6_arrays_13.json    20923273      True    556.139098   
9   depth_7_members_6_arrays_14.json    24975396      True    679.214590   
10  depth_7_members_6_arrays_15.json    27981828      True    737.743108   
11  depth_7_members_6_arrays_17.json    29378502      True    812.584832   
12  depth_7_

# Combined Rejection Plot (DFA vs. Regex vs. Similar-Size Accept)

This section produces a single three-way comparison from a pre-aggregated `reject.csv` (rather than merging raw per-acceptor CSVs as in the previous sections). The three series are:
- **DFA Reject** — DFA-based acceptor's time to reject an invalid file.
- **Regex Reject** — regex-based acceptor's time to reject the same file.
- **Similar-size DFA-Accept** — the DFA acceptor's time to *accept* a valid file of comparable size.

Plotting these together highlights the benefit of the DFA acceptor's early-abort behavior: rejection should be markedly faster than accepting a similarly-sized valid file, since accepting requires reading the whole file while rejection can stop the moment a rule violation is detected. The plot is saved as `plot_all_reject.png` / `.pdf`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/reject.csv')
df['size_mb'] = df['size_bytes'] / (1024 ** 2)
df = df.sort_values('size_mb').reset_index(drop=True)

long_df = df.melt(
    id_vars=['file', 'size_bytes', 'accepted', 'size_mb'],
    value_vars=['time_ms_dfa', 'time_ms_regex', 'similar_size_accepted'],
    var_name='method', value_name='time_ms'
)
label_map = {
    'time_ms_dfa': 'DFA Reject',
    'time_ms_regex': 'Regex Reject',
    'similar_size_accepted': 'Similar-size DFA-Accept',
}
long_df['method'] = long_df['method'].map(label_map)
long_df = long_df.sort_values('size_mb')

sns.set_theme(context='paper', style='whitegrid', font_scale=1.3)
plt.rcParams.update({
    'font.family': 'serif',
    'axes.edgecolor': '0.3',
    'axes.linewidth': 1.0,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.4,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'savefig.dpi': 300,
    'figure.dpi': 150,
})
palette = {
    'DFA Reject': '#1f77b4',
    'Regex Reject': '#d62728',
    'Similar-size DFA-Accept': '#2ca02c',
}
markers = {
    'DFA Reject': 'o',
    'Regex Reject': 'X',
    'Similar-size DFA-Accept': '^',
}

fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(
    data=long_df, x='size_mb', y='time_ms', hue='method',
    style='method', palette=palette, markers=markers, dashes=False,
    markersize=7, linewidth=1.5, alpha=0.9, ax=ax,
    markeredgecolor='white', markeredgewidth=0.6,
)
ax.set_xlabel('File size (MB)')
ax.set_ylabel('Time (ms)')
ax.set_title('Time vs. File Size', fontsize=13, weight='bold')
ax.legend(title='Method', loc='upper left')
fig.tight_layout()
fig.savefig('/content/plot_all_reject.png', bbox_inches='tight')
fig.savefig('/content/plot_all_reject.pdf', bbox_inches='tight')
plt.close(fig)
print('Saved plot_all_reject')

Saved plot_all_reject


# Throughput

Runtime alone doesn't account for how much work an acceptor actually did — a large file with few JSON tokens is "easier" than a smaller file packed with many symbols. This section normalizes timing by the number of grammar symbols processed (`total_symbols`, from a separately-computed `json_symbol_counts.csv`) to get a **throughput** metric (symbols processed per millisecond), which is a fairer basis for comparing the DFA-based and regex-based acceptors' raw processing speed.

**What this cell does:**
1. Re-merges the DFA and regex acceptance benchmarks (same as the Overall Experiment cell) into `merged_benchmark.csv`.
2. Loads `json_symbol_counts.csv` — a table with `total_symbols` per file (e.g. the count of `object` / `array` / `value` / `document` events encountered while parsing), and inner-joins it to the benchmark on `file` + `size_bytes`.
3. Computes `throughput_dfa` and `throughput_acceptor` as `total_symbols / time_ms` for each method (guarding against divide-by-zero via `.replace(0, np.nan)`).
4. Reshapes to long format and produces two log-log plots: **Total Symbols vs. File Size** (`plot_total_symbols.png/.pdf`, sanity-checking that symbol count scales with file size) and **Throughput vs. File Size** (`plot_throughput.png/.pdf`, the main result — showing whether either acceptor's symbols/ms rate degrades, stays flat, or improves as files grow).

**Note:** update `symbols_file_path` at the top of the cell to point to your actual `json_symbol_counts.csv` location before running.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# --- User input needed ---
symbols_file_path = '/content/json_symbol_counts.csv' # <<< IMPORTANT: Update this path

dfa = pd.read_csv('/content/hedge_dfa_acceptor_benchmark.csv')
acc = pd.read_csv('/content/hedge_acceptor_benchmark.csv')

key_cols = ['file', 'size_bytes', 'accepted']
dfa = dfa.rename(columns={'time_ms': 'time_ms_dfa'})
acc = acc.rename(columns={'time_ms': 'time_ms_acceptor'})

merged = pd.merge(dfa, acc, on=key_cols, how='inner', validate='one_to_one')
merged['size_mb'] = merged['size_bytes'] / (1024 ** 2)
merged = merged.sort_values('size_mb').reset_index(drop=True)

merged.to_csv('merged_benchmark.csv', index=False)
# --------------------------

# Load the total_symbols data
try:
    symbols_df = pd.read_csv(symbols_file_path)
except FileNotFoundError:
    print(f"Error: The file '{symbols_file_path}' was not found. Please update the path.")
    raise

# Assuming 'merged_benchmark.csv' from the 'Overall experiment' cell is available
# Or, if that cell hasn't been run, we can re-create 'merged' from individual files
# For simplicity, let's assume 'merged_benchmark.csv' is saved and load it.
# If it's not saved, the user would need to run the 'Overall experiment' first.
try:
    benchmark_df = pd.read_csv('merged_benchmark.csv')
except FileNotFoundError:
    print("Error: 'merged_benchmark.csv' not found. Please run the 'Overall experiment' cell first to generate this file.")
    raise

# Merge the dataframes
# We'll use 'file' and 'size_bytes' as common keys for merging.
# Ensure 'file' and 'size_bytes' columns are consistent in both dataframes.
merged_symbols_df = pd.merge(symbols_df, benchmark_df,
                             on=['file', 'size_bytes'],
                             how='inner', validate='one_to_one')

# Calculate throughput for both methods
# Ensure time_ms is not zero to avoid division by zero
merged_symbols_df['throughput_dfa'] = merged_symbols_df['total_symbols'] / merged_symbols_df['time_ms_dfa'].replace(0, np.nan)
merged_symbols_df['throughput_acceptor'] = merged_symbols_df['total_symbols'] / merged_symbols_df['time_ms_acceptor'].replace(0, np.nan)

# Convert to long format for plotting
long_throughput_df = merged_symbols_df.melt(
    id_vars=['file', 'size_bytes', 'total_symbols', 'size_mb'],
    value_vars=['throughput_dfa', 'throughput_acceptor'],
    var_name='method',
    value_name='throughput'
)
label_map_throughput = {'throughput_dfa': 'Dfa-based', 'throughput_acceptor': 'Regex-based'}
long_throughput_df['method'] = long_throughput_df['method'].map(label_map_throughput)

# Also prepare total_symbols for plotting (long format, though not strictly necessary for this plot type)
long_symbols_plot_df = merged_symbols_df.melt(
    id_vars=['file', 'size_bytes', 'size_mb'],
    value_vars=['total_symbols'],
    var_name='metric',
    value_name='value'
)

# --- Plotting setup (reusing previous style) ---
sns.set_theme(context='paper', style='whitegrid', font_scale=1.3)
plt.rcParams.update({
    'font.family': 'serif',
    'axes.edgecolor': '0.3',
    'axes.linewidth': 1.0,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.4,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'savefig.dpi': 300,
    'figure.dpi': 150,
})
palette = {'Dfa-based': '#1f77b4', 'Regex-based': '#d62728'}

# --- Plot 1: Total Symbols vs. File Size ---
fig1, ax1 = plt.subplots(figsize=(7, 5))
sns.lineplot(
    data=merged_symbols_df.sort_values('size_mb'),
    x='size_mb', y='total_symbols',
    markers=True, dashes=False, markersize=5, linewidth=1.3, alpha=0.9, ax=ax1,
    markeredgecolor='white', markeredgewidth=0.4,
    color='purple' # Single color as there's only one 'total_symbols' metric
)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('File size (MB, log scale)')
ax1.set_ylabel('Total Symbols (log scale)')
ax1.set_title('Total Symbols vs. File Size', fontsize=13, weight='bold')

# Custom formatter for x-axis if needed
def _fmt(x, pos):
    if x >= 1:
        return f'{x:g}'
    return f'{x:g}'
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt))
ax1.xaxis.set_minor_formatter(mticker.NullFormatter())

fig1.tight_layout()
fig1.savefig('plot_total_symbols.png', bbox_inches='tight')
fig1.savefig('plot_total_symbols.pdf', bbox_inches='tight')
plt.close(fig1)
print('Saved plot_total_symbols')

# --- Plot 2: Throughput vs. File Size ---
fig2, ax2 = plt.subplots(figsize=(7, 5))
sns.lineplot(
    data=long_throughput_df.sort_values('size_mb'),
    x='size_mb', y='throughput', hue='method',
    style='method', palette=palette, markers=True, dashes=False,
    markersize=5, linewidth=1.3, alpha=0.9, ax=ax2,
    markeredgecolor='white', markeredgewidth=0.4,
)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('File size (MB, log scale)')
ax2.set_ylabel('Throughput (Symbols/ms, log scale)')
ax2.set_title('Throughput vs. File Size', fontsize=13, weight='bold')
ax2.legend(title='Method', loc='upper left')

# Custom formatter for x-axis if needed
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt))
ax2.xaxis.set_minor_formatter(mticker.NullFormatter())

fig2.tight_layout()
fig2.savefig('plot_throughput.png', bbox_inches='tight')
fig2.savefig('plot_throughput.pdf', bbox_inches='tight')
plt.close(fig2)
print('Saved plot_throughput')

print('\nDone plotting symbols and throughput.')

Saved plot_total_symbols
Saved plot_throughput

Done plotting symbols and throughput.


### Mount Google Drive

This notebook is written for Google Colab. This cell mounts the user's Google Drive at `/content/drive` so that benchmark CSVs stored there (rather than uploaded directly to the Colab session) can be read by the cells above. Skip this cell if you're running the notebook locally or your CSVs already live under `/content/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Peak Throughput Summary

Reports the single highest throughput value (symbols/ms) observed across both acceptors and all files in `long_throughput_df` (computed in the Throughput section above), along with the full row of metadata (file, method, size, etc.) for that best case — useful for headline numbers in a write-up.

In [ ]:
max_throughput = long_throughput_df['throughput'].max()
print(f"Max Throughput: {max_throughput:.2f} Symbols/ms")

max_throughput_row = long_throughput_df.loc[long_throughput_df['throughput'].idxmax()]
print("Details for max throughput:\n")
print(max_throughput_row)

Max Throughput: 4175.85 Symbols/ms
Details for max throughput:

file               huge.json
size_bytes        2654148612
total_symbols      263742278
size_mb          2531.193363
method             Dfa-based
throughput       4175.850705
Name: 3, dtype: object


### Throughput Distribution Summary

Computes the mean, median, and standard deviation of throughput across `long_throughput_df`, giving a sense of the typical (not just best-case) performance and how much it varies across files and methods.

In [ ]:
mean_throughput = long_throughput_df['throughput'].mean()
median_throughput = long_throughput_df['throughput'].median()
std_throughput = long_throughput_df['throughput'].std()

print(f"Mean Throughput: {mean_throughput:.2f} Symbols/ms")
print(f"Median Throughput: {median_throughput:.2f} Symbols/ms")
print(f"Standard Deviation of Throughput: {std_throughput:.2f} Symbols/ms")

Mean Throughput: 3781.48 Symbols/ms
Median Throughput: 3869.35 Symbols/ms
Standard Deviation of Throughput: 686.24 Symbols/ms
